In [1]:
import glob
import os
import shutil
import json

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm

from tflite_inference import TFLiteInferer

2025-03-19 16:20:28.382007: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742394028.393783  335092 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742394028.397331  335092 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-19 16:20:28.409640: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version:  2.18.0


In [2]:
# !pip install tensorflow==2.3

In [3]:
def awb(img):
    # Improve contrast using histogram equalization (adaptive for better results)
    lab = cv2.cvtColor(img.copy(), cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    enhanced_image = cv2.merge((l, a, b))
    enhanced_image = cv2.cvtColor(enhanced_image, cv2.COLOR_LAB2RGB)
    return enhanced_image

In [4]:
def white_balance(img):
    result = cv2.cvtColor(img.copy(), cv2.COLOR_RGB2LAB)
    avg_a = np.average(result[:, :, 1])
    avg_b = np.average(result[:, :, 2])
    result[:, :, 1] = result[:, :, 1] - ((avg_a - 128) * (result[:, :, 0] / 255.0) * 1.1)
    result[:, :, 2] = result[:, :, 2] - ((avg_b - 128) * (result[:, :, 0] / 255.0) * 1.1)
    result = cv2.cvtColor(result, cv2.COLOR_LAB2RGB)
    return result

In [5]:
BASELINE_CONF_TH = 0.35
EXP_CONF_TH = 0.3
IOU_TH = 0.5

ORIGINAL_H, ORIGIAL_W = 480, 640
W = H = 320
RATIO = W / ORIGIAL_W

inferer_baseline = TFLiteInferer(
    yolo_pth="results/filtered_v3_margin_15/weights/best_saved_model/best_full_integer_quant.tflite",
    head_pth="results/filtered_v3_margin_15/weights/head_saved_model/head_float32.tflite",
    conf_th=BASELINE_CONF_TH,
    iou_th=IOU_TH,
    imgsz=320,
)

inferer_exp = TFLiteInferer(
    yolo_pth="results/filtered_v3_margin_15_reduced_quality/weights/best_saved_model/best_full_integer_quant.tflite",
    head_pth="results/filtered_v3_margin_15_reduced_quality/weights/head_saved_model/head_float32.tflite",
    conf_th=EXP_CONF_TH,
    iou_th=IOU_TH,
    imgsz=320,
)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [6]:
def draw(img, detections, title=""):
    tmp = img.copy()
    
    cv2.putText(tmp, title, (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    
    for box in detections:
        cv2.rectangle(tmp, (box[:2] / RATIO).astype(int), ((box[:2].astype(int) + box[2:4]) / RATIO).astype(int) , (255, 0, 0), 1)
        cv2.putText(tmp, f"{box[4]:.2f}", (box[:2] / RATIO).astype(int) + [0, 15], cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

    for p in detections:
        cv2.circle(tmp, (p[-2:] // RATIO).astype(int), 2, (255, 0, 0), 3)
    

    # plt.figure(figsize=(8, 8))
    # plt.imshow(tmp)
    return tmp

In [14]:
ipaths = glob.glob("/home/kyamkovyi/Pictures/cam4/250Hstanding4ppl/*.tif") + \
    glob.glob("/home/kyamkovyi/Pictures/cam4/New Folder/*.tif") + \
    glob.glob("/home/kyamkovyi/Pictures/cam4/New Folder/50/*.tif") + \
    glob.glob("/home/kyamkovyi/Pictures/cam4/New Folder/55/*.tif")
len(ipaths)

192

In [15]:
res_path = "/home/kyamkovyi/Videos/cam4_new_lens.avi"

results = []
for ipath in tqdm(ipaths):
    frame = cv2.imread(ipath)
    h, w, _ = frame.shape
    if h != 480 or w != 640:
        print(ipath)
        continue
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    # frame_awb = awb(frame)
    # frame_awb = white_balance(frame)
    # full_frame = np.zeros((480 * 2, 640 * 2, 3), dtype=np.uint8)
    full_frame = np.zeros((480, 640 * 2, 3), dtype=np.uint8)
    
    dets = inferer_baseline.infer(frame)
    res = draw(frame, dets, "baseline model")
    full_frame[:, :640] = res
    
    # dets = inferer_baseline.infer(frame_awb)
    # res = draw(frame_awb, dets, "baseline model + awb")
    # full_frame[:480, 640:] = res
    
    dets = inferer_exp.infer(frame)
    res = draw(frame, dets, "blurred model")
    full_frame[:, 640:] = res
    
    # dets = inferer_exp.infer(frame_awb)
    # res = draw(frame_awb, dets, "blurred model + awb")
    # full_frame[480:, 640:] = res
    
    results.append(full_frame)
    # break

cap = cv2.VideoWriter(res_path, cv2.VideoWriter_fourcc(*'DIVX'), 1, (640 * 2, 480))
for res in results:
    cap.write(cv2.cvtColor(res, cv2.COLOR_RGB2BGR))
cap.release()

100%|██████████| 192/192 [00:09<00:00, 20.87it/s]


In [21]:
frame.shape

(480, 640, 3)